# 0.1. Przygotowanie środowiska

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve() / "src"))

from config.config import setup

cfg = setup()

HAR_ROOT = cfg["HAR_ROOT"]
MODELS_DIR = cfg["MODELS_DIR"]
RESULTS_DIR = cfg["RESULTS_DIR"]
FIGURES_DIR = cfg["FIGURES_DIR"]
PROJECT_ROOT = cfg["PROJECT_ROOT"]

# 0.2. Wgranie danych z datasetu

In [ ]:
from data.loader import load_har_data

X_train, y_train, groups_train, X_test, y_test, groups_test, feature_names = (
    load_har_data(HAR_ROOT)
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
# X_test wczytany dla kompletności - nie dotykamy go przed Fazą 6

# 0.3. Kontekst Fazy 5

Faza 5 jest dla formalności - nie ma tu żadnego trenowania ani predykcji, po prostu wybranie modeli, które będą testowane.
Modele, które będą testowane są wytrenowane w fazie 3 lub w 4 (grid search cv miał refit=true).

# 1. Ładowanie modeli końcowych z Fazy 3

In [ ]:
import joblib

MODELS_SUBDIR = "03_phase"

MODELS_TO_EVALUATE = [
    "dummy",
    "logreg",
    "linear_svc",
    "rbf_svc",
    "random_forest",
    "gaussian_nb",
]

final_models = {}
for name in MODELS_TO_EVALUATE:
    grid_path = MODELS_DIR / f"{MODELS_SUBDIR}_{name}" / f"grid_{name}.joblib"
    grid = joblib.load(grid_path)
    final_models[name] = {
        "estimator": grid.best_estimator_,
        "best_params": grid.best_params_,
        "cv_score": grid.best_score_,
        "grid_path": str(grid_path),
    }
    print(f"[{name}] CV-MCC = {grid.best_score_:.4f}, params = {grid.best_params_}")

# 2. Sanity check - czy modele ok do predict

Dla każdego:
- Czy klasyfikator zna klasy (`classes_` - ustawiane przez `fit`).
- Czy scaler (jeśli istnieje w pipeline) jest dopasowany (`mean_`).

In [ ]:
all_ok = True

for name, info in final_models.items():
    estimator = info["estimator"]
    steps = dict(estimator.steps)
    errors = []

    if not hasattr(steps["clf"], "classes_"):
        errors.append("clf nie ma classes_ (nie wytrenowany)")

    if "scaler" in steps and not hasattr(steps["scaler"], "mean_"):
        errors.append("scaler nie ma mean_ (nie dopasowany)")

    if errors:
        print(f"[BLAD] {name}: {', '.join(errors)}")
        all_ok = False
    else:
        print(f"[OK]   {name}: gotowy do predict()")

if all_ok:
    print("\nWszystkie modele przeszly sanity check.")

# 3. Podsumowanie finalnych modeli

In [ ]:
import pandas as pd

rows = []
for name, info in final_models.items():
    estimator = info["estimator"]
    rows.append(
        {
            "model": name,
            "cv_mcc": round(info["cv_score"], 4),
            "best_params": str(info["best_params"]),
            "n_steps": len(estimator.steps),
            "clf_class": type(estimator.named_steps["clf"]).__name__,
            "grid_path": info["grid_path"],
        }
    )

df_final = (
    pd.DataFrame(rows).sort_values("cv_mcc", ascending=False).reset_index(drop=True)
)
print(df_final.to_string(index=False))

SUMMARY_DIR = RESULTS_DIR / "05_phase_final_models"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
df_final.to_csv(SUMMARY_DIR / "final_models_summary.csv", index=False)
print(f"\nZapisano: {SUMMARY_DIR / 'final_models_summary.csv'}")

# 4. Zapis indeksu modeli końcowych (dla porządku, podstawka do fazy 6)

Tworzony jest `final_models_index.json` ze ścieżkami względem `PROJECT_ROOT`, które wskazują na modele do użycia.

In [ ]:
import json

INDEX_DIR = MODELS_DIR / "05_phase_final_models"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

index = {
    name: str(Path(info["grid_path"]).relative_to(PROJECT_ROOT))
    for name, info in final_models.items()
}

index_path = INDEX_DIR / "final_models_index.json"
with open(index_path, "w") as f:
    json.dump(index, f, indent=2)

print(f"Zapisano manifest: {index_path}")
print(json.dumps(index, indent=2))